In [ ]:
import os
from pathlib import Path

from dotenv import load_dotenv
import pandas as pd
from premise import *
from premise import __version__, __file__

load_dotenv()
premise_key = os.environ["PREMISE_KEY"]
iam_files_dir = Path(os.environ.get("PREMISE_IAM_FILES_DIR", "iam_output_files"))
iam_files_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
print(__version__)

In [ ]:
print(__file__)

In [ ]:
import bw2data
bw2data.projects.set_current("ecoinvent-3.12-cutoff")

In [ ]:
ndb = NewDatabase(
    scenarios=[{**scenario, "filepath": iam_files_dir} for scenario in [
        {"model": "remind", "pathway": "SSP1-NPi", "year": 2025,},
        {"model": "remind", "pathway": "SSP1-PkBudg650", "year": 2025,},
        {"model": "remind", "pathway": "SSP1-PkBudg1000", "year": 2025,},
        {"model": "remind", "pathway": "SSP2-NDC", "year": 2025,},
        {"model": "remind", "pathway": "SSP2-NPi", "year": 2025,},
        {"model": "remind", "pathway": "SSP2-PkBudg650", "year": 2025,},
        {"model": "remind", "pathway": "SSP3-rollBack", "year": 2025,},
        {"model": "remind", "pathway": "SSP2-PkBudg1000", "year": 2025,},

        {"model": "remind-eu", "pathway": "SSP2-NDC", "year": 2025,},
        {"model": "remind-eu", "pathway": "SSP2-NPi", "year": 2025,},
        {"model": "remind-eu", "pathway": "SSP2-PkBudg650", "year": 2025,},
        {"model": "remind-eu", "pathway": "SSP2-PkBudg1000", "year": 2025,},

        {"model": "image", "pathway": "SSP1-L", "year": 2025,},
        {"model": "image", "pathway": "SSP1-M", "year": 2025,},
        {"model": "image", "pathway": "SSP1-VLLO", "year": 2025,},
        {"model": "image", "pathway": "SSP2-L", "year": 2025,},
        {"model": "image", "pathway": "SSP2-M", "year": 2025,},
        {"model": "image", "pathway": "SSP2-VLHO", "year": 2025,},
        {"model": "image", "pathway": "SSP3-H", "year": 2025,},
        {"model": "image", "pathway": "SSP5-H", "year": 2025,},

        {"model": "tiam-ucl", "pathway": "SSP2-Base", "year": 2025,},
        {"model": "tiam-ucl", "pathway": "SSP2-RCP19", "year": 2025,},
        {"model": "tiam-ucl", "pathway": "SSP2-RCP26", "year": 2025,},
        {"model": "tiam-ucl", "pathway": "SSP2-RCP45", "year": 2025,},

        #{"model": "gcam", "pathway": "SSP2-Base", "year": 2025,},
        #{"model": "gcam", "pathway": "SSP2-RCP26", "year": 2025,},
        #{"model": "gcam", "pathway": "SSP2-RCP45", "year": 2025,},

        {"model": "message", "pathway": "SSP1-L", "year": 2050},
        {"model": "message", "pathway": "SSP1-VL", "year": 2050},
        {"model": "message", "pathway": "SSP2-L", "year": 2050},
        {"model": "message", "pathway": "SSP2-LO", "year": 2050},
        {"model": "message", "pathway": "SSP2-M", "year": 2050},
        {"model": "message", "pathway": "SSP2-ML", "year": 2050},
        {"model": "message", "pathway": "SSP2-VL", "year": 2050},
        {"model": "message", "pathway": "SSP3-H", "year": 2050},
        {"model": "message", "pathway": "SSP4-LO", "year": 2050},
        {"model": "message", "pathway": "SSP5-H", "year": 2050},
        {"model": "message", "pathway": "SSP5-LO", "year": 2050},
    ]],
    source_db="ecoinvent-3.12-cutoff", # <-- name of the database in the BW2 project. Must be a string.
    source_version="3.12", # <-- version of ecoinvent. Can be "3.5", "3.6", "3.7" or "3.8". Must be a string.
    key=premise_key,
    biosphere_name="biosphere",
)

In [ ]:
import pandas as pd

# Load the Excel file to check all sheet names
file_path = 'mapping_overview.xlsx'
excel_file = pd.ExcelFile(file_path)

# Display all sheet names
sheet_names = excel_file.sheet_names

# Load data from each sheet into a dictionary of dataframes
dfs = {sheet: pd.read_excel(file_path, sheet_name=sheet) for sheet in sheet_names}

# Concatenate all the dataframes into one, adding a column to indicate the source sheet
combined_df = pd.concat([df.assign(Sheet=sheet) for sheet, df in dfs.items()], ignore_index=True)


In [ ]:
combined_df = combined_df.loc[combined_df["Variable Type"]=="Production volume"]

In [ ]:
len(combined_df)

In [ ]:
large_data = pd.DataFrame()
for scenario in ndb.scenarios:
    model = scenario["model"]
    pathway = scenario["pathway"]
    print(model, pathway)
    model_variables = combined_df.loc[
        combined_df["IAM Model"] == model, ["Sheet", "Key"]
    ].drop_duplicates()
    for (sector, var), row in model_variables.groupby(
        ["Sheet", "Key"]
    ):
        
        data = None

        if var in (
            "CO2", "gdp", "population", "GMST"
        ):
            data = scenario["iam data"].other_vars.sel(variables=var)
        else:
            if var in scenario["iam data"].production_volumes.coords["variables"].values.tolist():
                data = scenario["iam data"].production_volumes.sel(variables=var)
            else:
                continue

        if data is not None:
            long_data = data.to_dataframe("val").reset_index()
            long_data["sector"] = sector
            long_data["model"] = model
            long_data["scenario"] = pathway
            long_data = long_data.loc[long_data["year"] <= 2100]
            large_data = pd.concat([large_data, long_data])

In [ ]:
large_data.loc[
    large_data["sector"] == "Carbon Dioxide Removal",
    "variables"
].unique()

In [ ]:
sectors = large_data["sector"].unique().tolist()
for sector in sectors:
    print(sector)


In [ ]:
len(large_data)

In [ ]:
large_data.loc[
    (large_data["sector"]=="Carbon Dioxide Removal")
    &(large_data["val"]<0), 
    "val"
] *= -1

large_data.loc[large_data["variables"]=="population", "sector"] = "Population"
large_data.loc[large_data["variables"]=="CO2", "sector"] = "Carbon Dioxide emissions"
large_data.loc[large_data["variables"]=="gdp", "sector"] = "Gross Domestic Product"
large_data.loc[large_data["variables"]=="GMST", "sector"] = "GMST increase"

large_data.loc[
    (large_data["variables"].str.contains("diesel"))
    &(~large_data["variables"].str.contains("truck"))
    &(~large_data["variables"].str.contains("train"))
    &(~large_data["variables"].str.contains("bus"))
    &(~large_data["variables"].str.contains("car"))
    &(~large_data["variables"].str.contains("ship"))
    , "sector"] = "Diesel"

large_data.loc[large_data["variables"].str.contains("ethanol"), "sector"] = "Gasoline"
large_data.loc[
    (large_data["variables"].str.contains("gasoline"))
    &(~large_data["variables"].str.contains("truck"))
    &(~large_data["variables"].str.contains("train"))
    &(~large_data["variables"].str.contains("bus"))
    &(~large_data["variables"].str.contains("car"))
    &(~large_data["variables"].str.contains("two-wheeler"))
    , "sector"] = "Gasoline"

large_data.loc[large_data["variables"].str.contains("liquefied"), "sector"] = "LPG"
large_data.loc[large_data["variables"].str.contains("kerosene"), "sector"] = "Kerosene"
large_data.loc[large_data["variables"].str.contains("hydrogen"), "sector"] = "Hydrogen"

large_data.loc[large_data["variables"]=="natural gas", "sector"] = "Gas"
large_data.loc[large_data["variables"]=="biomethane", "sector"] = "Gas"
large_data.loc[large_data["variables"]=="heavy fuel oil", "sector"] = "Oil"

large_data = large_data.replace("truck, ", "", regex=True)
large_data = large_data.replace("train, ", "", regex=True)
large_data = large_data.replace("bus, ", "", regex=True)
large_data = large_data.replace("passenger car, ", "", regex=True)
large_data = large_data.replace("two-wheeler, ", "", regex=True)

large_data = large_data.replace(", mini", "", regex=True)
large_data = large_data.replace(", medium SUV", "", regex=True)
large_data = large_data.replace(", large SUV", "", regex=True)
large_data = large_data.replace(", medium", "", regex=True)
large_data = large_data.replace(", van", "", regex=True)
large_data = large_data.replace(", large", "", regex=True)
large_data = large_data.replace(", small", "", regex=True)

large_data = large_data.replace(", 3.5t", "", regex=True)
large_data = large_data.replace(", 7.5t", "", regex=True)
large_data = large_data.replace(", 18t", "", regex=True)
large_data = large_data.replace(", 26t", "", regex=True)
large_data = large_data.replace(", 40t", "", regex=True)

large_data = large_data.replace(", energy allocation", "", regex=True)
large_data = large_data.replace("kerosene, ", "", regex=True)
large_data = large_data.replace(", PEM", "", regex=True)
large_data = large_data.replace("hydrogen, ", "", regex=True)
large_data = large_data.replace("liquefied petroleum gas, ", "LPG", regex=True)
large_data = large_data.replace("steel - ", "", regex=True)

large_data = large_data.groupby([
    "region", "year", "variables", "sector", "model", "scenario"
]).sum().reset_index()


In [ ]:
large_data.loc[
    (large_data["model"].isin(["image", "remind", "remind-eu", "gcam"]))
    &(large_data["variables"]=="CO2"),
    "val"
] *= 1000000
large_data.loc[
    (large_data["model"]=="tiam-ucl")
    &(large_data["variables"]=="CO2"),
    "val"
] *= 1000
large_data.loc[
    (large_data["model"]=="tiam-ucl")
    &(large_data["variables"]=="gdp"),
    "val"
] *= 10000

large_data.loc[
    (large_data["model"]=="tiam-ucl")
    &(large_data["variables"].isin(['biomass crops - purpose grown', 'biomass wood - purpose grown', 'biomass - residual'])),
    "val"
] /= 1000

large_data.loc[
    (large_data["model"]=="remind")
    &(large_data["variables"].isin(['biomass crops - purpose grown', 'biomass wood - purpose grown', 'biomass - residual'])),
    "val"
] *= 10

large_data.loc[
    (large_data["model"] == "tiam-ucl")
    &(large_data["sector"]=="Carbon Dioxide Removal"),
    "val"
] /= 1000

In [ ]:
is_heat_sector = large_data["sector"] == "Heat"
is_heat_variable = large_data["variables"].str.startswith("heat", na=False)
large_data = large_data.loc[~is_heat_sector | is_heat_variable].copy()
large_data.loc[
    large_data["variables"].str.startswith("heat, ", na=False), "sector"
] = "Heat"

In [ ]:
large_data = large_data.loc[large_data["val"] != 0]
large_data = large_data.loc[large_data["val"].abs()>0.1]

In [ ]:
len(large_data)

In [ ]:
len(large_data)

In [ ]:
fp = f"../data/structured_data {str(__version__)}.csv"
large_data.to_csv(fp)